# Preparation potential training and occurrence of ecosystems (res. 0.01°)

In [ ]:
# Libraries
import os
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'

---

In [ ]:
# Get data
# Get copernicus data
ds_cop = xr.open_zarr(f'{dir01}ds_prep_cop_r001.zarr')
# Get hilda data
ds_hilda = xr.open_zarr(f'{dir01}ds_prep_hilda_stable_r001.zarr')
# Get mu et al hfp data
ds_mu = xr.open_zarr(f'{dir01}ds_prep_mu_hfpmax_r001.zarr')
# Get dinerstein et al data
ds_din = xr.open_zarr(f'{dir01}ds_prep_dinerstein_biomes_r001.zarr')
# Get land mask
ds_land = xr.open_zarr(f'{dir01}ds_prep_copernicus_land_mask_r001.zarr')

# Dinerstein et al categories
# 1	 Tropical & Subtropical Moist Broadleaf Forests
# 2	 Tropical & Subtropical Dry Broadleaf Forests
# 3	 Tropical & Subtropical Coniferous Forests
# 4	 Temperate Broadleaf & Mixed Forests
# 5	 Temperate Conifer Forests
# 6	 Boreal Forests/Taiga
# 7	 Tropical & Subtropical Grasslands, Savannas & ...
# 8	 Temperate Grasslands, Savannas & Shrublands
# 9	 Flooded Grasslands & Savannas
# 10 Montane Grasslands & Shrublands
# 11 Tundra
# 12 Mediterranean Forests, Woodlands & Scrub
# 13 Deserts & Xeric Shrublands
# 14 Mangroves

In [ ]:
# Ecosystem categories
dict_etype = dict(trop_ef = 'Tropical/subtropical evergreen forest', 
                  trop_df = 'Tropical/subtropical deciduous forest',
                  temp_ebf = 'Temperate/boreal evergreen broadleaf forest',
                  temp_dbf = 'Temperate/boreal deciduous broadleaf forest',
                  temp_enf = 'Temperate/boreal evergreen needleleaf forest',
                  temp_dnf = 'Temperate/boreal deciduous needleleaf forest',
                  glob_shrub = 'Shrublands',
                  glob_grass = 'Grasslands',
                  glob_tundr = 'Tundra',
                  glob_mount = 'Mountain grasslands and shrublands',
                  glob_crop = 'Cropland',
                  glob_past = 'Pasture')

In [ ]:
# Dinnerstein summarized categories
# Check with 01_10_prep_dinerstein!
l_din_trop = [1, 2, 3, 7, 14]
l_din_temp = [4, 5, 6, 8, 12]
l_din_shrub = [1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14]
l_din_grass = [1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14]
l_din_tundr = [11]
l_din_mount = [10]

In [ ]:
# Prepare mu arrays
da_mu_0 = xr.where(ds_mu.mu_hfpmax == 0, True, False)
da_mu_less1 = xr.where(ds_mu.mu_hfpmax < 1, True, False)
da_mu_less2 = xr.where(ds_mu.mu_hfpmax < 2, True, False)
da_mu_less3 = xr.where(ds_mu.mu_hfpmax < 3, True, False)
da_mu_less4 = xr.where(ds_mu.mu_hfpmax < 4, True, False)
da_mu_less10 = xr.where(ds_mu.mu_hfpmax < 10, True, False)
da_mu_more4 = xr.where(ds_mu.mu_hfpmax > 4, True, False)

In [ ]:
# Select potential training grid cells for each ecosystem based on r001 data
def prep_rel_grid_cells(etype):
    #trop_ef
    if etype == 'trop_ef':
        da_din_sel = xr.where(ds_din.dinerstein_biomes.isin(l_din_trop), True, False)
        da_hilda_sel = xr.where(ds_hilda.hilda_stable_forest > .99, True, False)
        da_cop_sel = xr.where((ds_cop.cop_forest_ebf > 99) | 
                              (ds_cop.cop_forest_enf > 99), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 90, True, False)
        da_mu_sel = da_mu_less4

    #trop_df
    if etype == 'trop_df':
        da_din_sel = xr.where(ds_din.dinerstein_biomes.isin(l_din_trop), True, False)
        da_hilda_sel = xr.where(ds_hilda.hilda_stable_forest > .99, True, False)
        da_cop_sel = xr.where((ds_cop.cop_forest_dbf > 99) | 
                              (ds_cop.cop_forest_dnf > 99), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 90, True, False)
        da_mu_sel = da_mu_less4

    #temp_ebf
    if etype == 'temp_ebf':
        da_din_sel = xr.where(ds_din.dinerstein_biomes.isin(l_din_temp), True, False)
        da_hilda_sel = xr.where(ds_hilda.hilda_stable_forest > .99, True, False)
        da_cop_sel = xr.where((ds_cop.cop_forest_ebf > 99), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 90, True, False)
        da_mu_sel = da_mu_less4

    #temp_dbf
    if etype == 'temp_dbf': 
        da_din_sel = xr.where(ds_din.dinerstein_biomes.isin(l_din_temp), True, False)
        da_hilda_sel = xr.where(ds_hilda.hilda_stable_forest > .99, True, False)
        da_cop_sel = xr.where((ds_cop.cop_forest_dbf > 99), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 90, True, False)
        da_mu_sel = da_mu_less4

    #temp_enf
    if etype == 'temp_enf':
        da_din_sel = xr.where(ds_din.dinerstein_biomes.isin(l_din_temp), True, False)
        da_hilda_sel = xr.where(ds_hilda.hilda_stable_forest > .99, True, False)
        da_cop_sel = xr.where((ds_cop.cop_forest_enf > 99), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 90, True, False)
        da_mu_sel = da_mu_less4

    #temp_dnf
    if etype == 'temp_dnf':
        da_din_sel = xr.where(ds_din.dinerstein_biomes.isin(l_din_temp), True, False)
        da_hilda_sel = xr.where(ds_hilda.hilda_stable_forest > .99, True, False)
        da_cop_sel = xr.where((ds_cop.cop_forest_dnf > 99), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 90, True, False)
        da_mu_sel = da_mu_less4

    #glob_shrub
    if etype == 'glob_shrub':
        da_din_sel = xr.where(ds_din.dinerstein_biomes.isin(l_din_shrub), True, False)
        da_hilda_sel = xr.where(ds_hilda.hilda_stable_shrub > .99, True, False)
        da_cop_sel = xr.where((ds_cop.cop_shrub > 50), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree < 10, True, False)
        da_mu_sel = da_mu_less4

    #glob_grass
    if etype == 'glob_grass': 
        da_din_sel = xr.where(ds_din.dinerstein_biomes.isin(l_din_grass), True, False)
        da_hilda_sel = xr.where(ds_hilda.hilda_stable_shrub > .99, True, False)
        da_cop_sel = xr.where((ds_cop.cop_grass > 50), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree < 10, True, False)
        da_mu_sel = da_mu_less4

    #glob_tundr
    if etype == 'glob_tundr':
        da_din_sel = xr.where(ds_din.dinerstein_biomes.isin(l_din_tundr), True, False)
        da_hilda_sel = da_din_sel
        da_cop_sel = da_din_sel
        da_cop_sel2 = xr.where(ds_cop.cop_tree < 10, True, False)
        da_mu_sel = da_mu_less4

    #glob_mount
    if etype == 'glob_mount':
        da_din_sel = xr.where(ds_din.dinerstein_biomes.isin(l_din_mount), True, False)
        da_hilda_sel = da_din_sel
        da_cop_sel = da_din_sel
        da_cop_sel2 = xr.where(ds_cop.cop_tree < 10, True, False)
        da_mu_sel = da_mu_less4

    # glob crop
    if etype == 'glob_crop':
        da_din_sel = xr.ones_like(ds_din.dinerstein_biomes).astype('bool')
        da_hilda_sel = xr.where(ds_hilda.hilda_stable_crop > .99, True, False)
        da_cop_sel = xr.where((ds_cop.cop_crops > 90), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree < 10, True, False)
        da_mu_sel = da_mu_more4

    # glob_past
    if etype == 'glob_past':
        da_din_sel = xr.ones_like(ds_din.dinerstein_biomes).astype('bool')
        da_hilda_sel = xr.where(ds_hilda.hilda_stable_pasture > .99, True, False)
        da_cop_sel = xr.where((ds_cop.cop_grass > 90), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree < 10, True, False)
        da_mu_sel = da_mu_more4

    # create arrays with potential training grid cells    
    ds_etype = xr.Dataset()
    
    ds_etype['pot_train_' + etype] = xr.where((da_din_sel == True) &
                                              (da_hilda_sel == True) &
                                              (da_cop_sel == True) &
                                              (da_cop_sel2 == True) &
                                              (da_mu_sel == True),
                                              True, False)
    
    ds_etype['pot_occur_' + etype] = da_din_sel
    
    return ds_etype

In [ ]:
# Create dataset of potential training grid cells for each ecozone
ds_pot_train = xr.Dataset()

for etype in dict_etype.keys():
    ds_pot_train[etype] = prep_rel_grid_cells(etype)['pot_train_' + etype]
    
# Create dataset of potential occurance grid cells for each ecozone
ds_pot_occur = xr.Dataset()

for etype in dict_etype.keys():
    ds_pot_occur[etype] = prep_rel_grid_cells(etype)['pot_occur_' + etype]
    
# Export
ds_pot_train.to_zarr(f'{dir02}ds_pot_train_r001.zarr', mode='w')
ds_pot_occur.to_zarr(f'{dir02}ds_pot_occur_r001.zarr', mode='w');